In [ ]:
# Cell 1 — Imports

import os
import glob
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("Librosa:", librosa.__version__)


In [ ]:
# Cell 2 — Project paths
from pathlib import Path

# Base directory (steps up one level from 'notebooks/' to the repo root)
BASE_DIR = Path("..")

# Input data
DATASET_DIR = BASE_DIR / "data" / "raw"

# Processed features and image arrays
MEL_DIR = BASE_DIR / "data" / "processed" / "mel_spectrograms"
FEATURE_DIR = BASE_DIR / "data" / "processed" / "features"

# Saved models and weights
MODEL_DIR = BASE_DIR / "models"

# Execution outputs
RESULTS_DIR = BASE_DIR / "outputs" / "results"
FIGURES_DIR = BASE_DIR / "outputs" / "figures"

# Create directories if they do not exist
for folder in [MEL_DIR, FEATURE_DIR, MODEL_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("Mel output:", MEL_DIR)
print("Feature output:", FEATURE_DIR)
print("Model output:", MODEL_DIR)
print("Results output:", RESULTS_DIR)

In [ ]:
# Cell 3 — Find every WAV file

audio_files = sorted(
    glob.glob(str(DATASET_DIR / "**" / "*.wav"), recursive=True)
)

print("WAV files found:", len(audio_files))

if not audio_files:
    raise FileNotFoundError(
        f"No WAV files found under {DATASET_DIR}. Check DATASET_DIR in Cell 3."
    )

for path in audio_files[:10]:
    print(path)


In [ ]:
# Cell 4 — Create speaker and gender labels

# Expected structure:
# dataset/
#   amh-spk-1-M/
#       audio001.wav
#   amh-spk-2-F/
#       audio002.wav

df = pd.DataFrame({"audio_path": audio_files})
df["speaker"] = df["audio_path"].apply(lambda p: Path(p).parent.name)

def infer_gender(speaker):
    name = str(speaker).upper().strip()
    if name.endswith("-M"):
        return "Male"
    if name.endswith("-F"):
        return "Female"
    return "Unknown"

df["gender"] = df["speaker"].apply(infer_gender)

print("Number of speakers:", df["speaker"].nunique())
print("\nRecordings per speaker:")
print(df["speaker"].value_counts().sort_index())

print("\nGender counts:")
print(df["gender"].value_counts())

display(df.head())


In [ ]:
# Cell 5 — Inspect one audio file

sample_path = audio_files[0]
y, sr = librosa.load(sample_path, sr=None)

print("File:", sample_path)
print("Sample rate:", sr, "Hz")
print("Samples:", len(y))
print("Duration:", round(len(y) / sr, 2), "seconds")

plt.figure(figsize=(12, 4))
librosa.display.waveshow(y, sr=sr)
plt.title("Example Audio Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()


In [ ]:
# Cell 6 — Function to generate and save a Mel spectrogram PNG

def save_mel_spectrogram(
    audio_path,
    output_root,
    n_mels=128,
    n_fft=2048,
    hop_length=512
):
    """Convert one WAV file to a Mel spectrogram and save it as PNG."""

    y, sr = librosa.load(audio_path, sr=None)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    speaker = Path(audio_path).parent.name
    output_dir = Path(output_root) / speaker
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"{Path(audio_path).stem}.png"

    plt.figure(figsize=(8, 4))
    librosa.display.specshow(
        mel_db,
        sr=sr,
        hop_length=hop_length,
        x_axis="time",
        y_axis="mel"
    )
    plt.colorbar(format="%+2.0f dB")
    plt.title(Path(audio_path).name)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()

    return str(output_path)

print("Example saved to:", save_mel_spectrogram(sample_path, MEL_DIR))


In [ ]:
# Cell 7 — Generate a Mel spectrogram PNG for EVERY WAV file

generated_pngs = []
mel_errors = []

for i, audio_path in enumerate(audio_files, start=1):
    try:
        generated_pngs.append(
            save_mel_spectrogram(audio_path, MEL_DIR)
        )
    except Exception as e:
        mel_errors.append({
            "audio_path": audio_path,
            "error": str(e)
        })

    if i % 50 == 0 or i == len(audio_files):
        print(f"Processed {i}/{len(audio_files)}")

print("\nGenerated PNGs:", len(generated_pngs))
print("Errors:", len(mel_errors))

if mel_errors:
    pd.DataFrame(mel_errors).to_csv(
        RESULTS_DIR / "mel_generation_errors.csv",
        index=False
    )

In [ ]:
# Cell 8 — Preview one  generated Mel spectrogram

if generated_pngs:
    img = plt.imread(generated_pngs[0])

    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Generated Mel Spectrogram PNG")
    plt.show()

    print(generated_pngs[0])


In [ ]:
# Cell 9 — MFCC feature extraction

def extract_mfcc_features(
    audio_path,
    n_mfcc=13,
    n_fft=2048,
    hop_length=512
):
    """Return 26 fixed-length numerical features.

    13 MFCC coefficients are calculated over time.
    For each coefficient we calculate:
      - mean
      - standard deviation

    13 + 13 = 26 features.
    """

    y, sr = librosa.load(audio_path, sr=None)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length
    )

    mean = np.mean(mfcc, axis=1)
    std = np.std(mfcc, axis=1)

    return np.concatenate([mean, std])

example_features = extract_mfcc_features(sample_path)

print("Feature vector length:", len(example_features))
print("First values:", example_features[:10])


In [ ]:
# Cell 10 — Extract MFCC features from every WAV

rows = []
mfcc_errors = []

for i, row in df.iterrows():
    try:
        features = extract_mfcc_features(row["audio_path"])

        record = {
            "audio_path": row["audio_path"],
            "speaker": row["speaker"],
            "gender": row["gender"]
        }

        for j, value in enumerate(features, start=1):
            record[f"mfcc_{j}"] = float(value)

        rows.append(record)

    except Exception as e:
        mfcc_errors.append({
            "audio_path": row["audio_path"],
            "error": str(e)
        })

    if (i + 1) % 50 == 0 or (i + 1) == len(df):
        print(f"Processed {i + 1}/{len(df)}")

mfcc_df = pd.DataFrame(rows)

mfcc_csv = FEATURE_DIR / "mfcc_features.csv"
mfcc_df.to_csv(mfcc_csv, index=False)

print("\nMFCC table shape:", mfcc_df.shape)
print("Saved:", mfcc_csv)

if mfcc_errors:
    pd.DataFrame(mfcc_errors).to_csv(
        RESULTS_DIR / "mfcc_errors.csv",
        index=False
    )


In [ ]:
print("Number of speakers:", mfcc_df["speaker"].nunique())

print("\nRecordings per speaker:")
print(
    mfcc_df["speaker"]
    .value_counts()
    .sort_index()
)

print("\nSpeakers with fewer than 2 recordings:")

problem_speakers = (
    mfcc_df["speaker"]
    .value_counts()
    .loc[lambda x: x < 2]
)

print(problem_speakers)

In [ ]:
# Cell 12 — Clean speakers and create ONE shared train/test split

# Count recordings for every speaker
speaker_counts = mfcc_df["speaker"].value_counts()

# Show speakers with too few recordings
problem_speakers = speaker_counts[speaker_counts < 2]

print("Speakers with fewer than 2 recordings:")
print(problem_speakers)

# Keep only speakers that have at least 2 recordings
valid_speakers = speaker_counts[
    speaker_counts >= 2
].index

mfcc_df_clean = mfcc_df[
    mfcc_df["speaker"].isin(valid_speakers)
].copy()

print("\nOriginal recordings:", len(mfcc_df))
print("Usable recordings:", len(mfcc_df_clean))
print("Removed recordings:", len(mfcc_df) - len(mfcc_df_clean))

print("\nSpeakers used for training/testing:")
print(mfcc_df_clean["speaker"].value_counts().sort_index())


# ---------------------------------------------------------
# Create ONE shared train/test split
# ---------------------------------------------------------
# Both the MFCC and Mel Spectrogram pipelines will use
# these exact same audio files.
#
# This makes our comparison fair.
# ---------------------------------------------------------

train_df, test_df = train_test_split(
    mfcc_df_clean,
    test_size=0.20,
    random_state=42,
    stratify=mfcc_df_clean["speaker"]
)

print("\nTraining files:", len(train_df))
print("Testing files:", len(test_df))

print("\nTraining distribution:")
print(
    train_df["speaker"]
    .value_counts()
    .sort_index()
)

print("\nTesting distribution:")
print(
    test_df["speaker"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Cell 13 — Prepare MFCC training/testing matrices
# IMPORTANT: Use mfcc_df_clean because it contains only
# speakers with at least 2 recordings.

# Get the MFCC feature columns
mfcc_columns = [
    c for c in mfcc_df_clean.columns
    if c.startswith("mfcc_")
]

# Create label encoder using ONLY the 18 valid speakers
label_encoder = LabelEncoder()

label_encoder.fit(
    mfcc_df_clean["speaker"]
)

# Convert MFCC features into NumPy arrays
X_mfcc_train = train_df[mfcc_columns].values.astype(np.float32)
X_mfcc_test = test_df[mfcc_columns].values.astype(np.float32)

# Convert speaker names into numerical labels
y_mfcc_train = label_encoder.transform(
    train_df["speaker"]
)

y_mfcc_test = label_encoder.transform(
    test_df["speaker"]
)

# Display information
print("Training features:", X_mfcc_train.shape)
print("Testing features:", X_mfcc_test.shape)

print("\nNumber of speakers:", len(label_encoder.classes_))

print("\nSpeakers:")
for i, speaker in enumerate(label_encoder.classes_):
    print(i, "→", speaker)

In [ ]:
# Cell 14 — Train MFCC → Random Forest

mfcc_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

mfcc_model.fit(
    X_mfcc_train,
    y_mfcc_train
)

mfcc_pred = mfcc_model.predict(
    X_mfcc_test
)

mfcc_accuracy = accuracy_score(
    y_mfcc_test,
    mfcc_pred
)

mfcc_f1 = f1_score(
    y_mfcc_test,
    mfcc_pred,
    average="weighted"
)

print("MFCC + Random Forest")
print("Accuracy:", round(mfcc_accuracy, 4))
print("Weighted F1:", round(mfcc_f1, 4))

In [ ]:
# Cell 15 — MFCC evaluation

print(classification_report(
    y_mfcc_test,
    mfcc_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

cm = confusion_matrix(
    y_mfcc_test,
    mfcc_pred,
    labels=np.arange(len(label_encoder.classes_))
)

plt.figure(figsize=(10, 8))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    range(len(label_encoder.classes_)),
    label_encoder.classes_,
    rotation=90
)

plt.yticks(
    range(len(label_encoder.classes_)),
    label_encoder.classes_
)

plt.xlabel("Predicted speaker")
plt.ylabel("Actual speaker")
plt.title("MFCC + Random Forest Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 — Fixed-size Mel spectrogram arrays for CNN

MEL_SHAPE = (128, 128)

def extract_mel_array(
    audio_path,
    n_mels=128,
    target_width=128,
    n_fft=2048,
    hop_length=512
):
    """Return a normalized 128x128 Mel spectrogram with one channel."""

    y, sr = librosa.load(audio_path, sr=None)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Audio durations can differ, so resize the time axis.
    resized = tf.image.resize(
        mel_db[..., np.newaxis],
        [n_mels, target_width]
    ).numpy().squeeze()

    # Normalize each recording to 0–1.
    minimum = resized.min()
    maximum = resized.max()

    resized = (resized - minimum) / (maximum - minimum + 1e-8)

    return resized[..., np.newaxis].astype(np.float32)


def build_mel_dataset(split_df):
    X, y, paths, errors = [], [], [], []

    for _, row in split_df.iterrows():
        try:
            X.append(extract_mel_array(row["audio_path"]))
            y.append(label_encoder.transform([row["speaker"]])[0])
            paths.append(row["audio_path"])
        except Exception as e:
            errors.append({
                "audio_path": row["audio_path"],
                "error": str(e)
            })

    return np.array(X), np.array(y), paths, errors


X_mel_train, y_mel_train, train_mel_paths, train_mel_errors = build_mel_dataset(train_df)
X_mel_test, y_mel_test, test_mel_paths, test_mel_errors = build_mel_dataset(test_df)

print("Mel training shape:", X_mel_train.shape)
print("Mel testing shape:", X_mel_test.shape)
print("Errors:", len(train_mel_errors) + len(test_mel_errors))


In [ ]:
# Cell 17 — Build Mel Spectrogram → CNN

num_classes = len(label_encoder.classes_)

mel_model = models.Sequential([
    layers.Input(shape=(128, 128, 1)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Dropout(0.30),
    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),

    layers.Dense(num_classes, activation="softmax")
])

mel_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mel_model.summary()


In [ ]:
# Cell 18 — Train the Mel Spectrogram CNN

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = mel_model.fit(
    X_mel_train,
    y_mel_train,
    validation_split=0.20,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Cell 19 — Evaluate Mel Spectrogram CNN

mel_loss, mel_accuracy = mel_model.evaluate(
    X_mel_test,
    y_mel_test,
    verbose=0
)

mel_probabilities = mel_model.predict(X_mel_test, verbose=0)
mel_pred = np.argmax(mel_probabilities, axis=1)

mel_f1 = f1_score(
    y_mel_test,
    mel_pred,
    average="weighted"
)

print("Mel Spectrogram + CNN")
print("Accuracy:", round(mel_accuracy, 4))
print("Weighted F1:", round(mel_f1, 4))


In [ ]:
# Cell 20 — Mel CNN evaluation

print(classification_report(
    y_mel_test,
    mel_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

cm = confusion_matrix(y_mel_test, mel_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm)
plt.colorbar()
plt.xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=90)
plt.yticks(range(len(label_encoder.classes_)), label_encoder.classes_)
plt.xlabel("Predicted speaker")
plt.ylabel("Actual speaker")
plt.title("Mel Spectrogram + CNN Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 21 — Compare both pipelines and select the winner

comparison = pd.DataFrame({
    "Pipeline": [
        "MFCC + Random Forest",
        "Mel Spectrogram + CNN"
    ],
    "Accuracy": [
        mfcc_accuracy,
        mel_accuracy
    ],
    "Weighted F1": [
        mfcc_f1,
        mel_f1
    ]
})

display(comparison)

# Weighted F1 is the primary metric; accuracy breaks a tie.
if mel_f1 > mfcc_f1 or (
    np.isclose(mel_f1, mfcc_f1) and mel_accuracy > mfcc_accuracy
):
    BEST_PIPELINE = "mel"
else:
    BEST_PIPELINE = "mfcc"

print("BEST PIPELINE:", "Mel Spectrogram + CNN" if BEST_PIPELINE == "mel" else "MFCC + Random Forest")


In [ ]:
# Cell 22 — Save both models and the selected pipeline information

# Keeping both models makes the experiment reproducible.
joblib.dump(mfcc_model, MODEL_DIR / "mfcc_random_forest.joblib")
mel_model.save(MODEL_DIR / "mel_cnn.keras")
joblib.dump(label_encoder, MODEL_DIR / "speaker_label_encoder.joblib")

metadata = {
    "best_pipeline": BEST_PIPELINE,
    "speakers": list(label_encoder.classes_),
    "mel_shape": [128, 128],
    "n_mfcc": 13,
    "n_fft": 2048,
    "hop_length": 512
}

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

comparison.to_csv(
    RESULTS_DIR / "pipeline_comparison.csv",
    index=False
)

print("Saved final artifacts to:", MODEL_DIR)


## 🎤 Final Gradio Voice Recognition

The UI accepts an uploaded audio file or microphone recording.

It applies **exactly the same feature extraction used during training**, then returns:

- **Speaker:** e.g. `amh-spk-1-M`
- **Gender:** `Male` / `Female`
- **Confidence:** model probability

Gender is derived from the speaker label suffix (`-M` or `-F`).


In [ ]:
# Cell 23 — Final prediction function

saved_mfcc_model = joblib.load(MODEL_DIR / "mfcc_random_forest.joblib")
saved_mel_model = tf.keras.models.load_model(MODEL_DIR / "mel_cnn.keras")
saved_label_encoder = joblib.load(MODEL_DIR / "speaker_label_encoder.joblib")

def gender_from_speaker(speaker):
    name = str(speaker).upper().strip()
    if name.endswith("-M"):
        return "Male"
    if name.endswith("-F"):
        return "Female"
    return "Unknown"


def predict_speaker(audio_path):
    if audio_path is None:
        return "No audio provided", "Unknown", 0.0

    if BEST_PIPELINE == "mfcc":
        features = extract_mfcc_features(audio_path).reshape(1, -1)

        probabilities = saved_mfcc_model.predict_proba(features)[0]
        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])

    else:
        mel = extract_mel_array(audio_path)
        probabilities = saved_mel_model.predict(
            np.expand_dims(mel, axis=0),
            verbose=0
        )[0]

        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])

    speaker = saved_label_encoder.inverse_transform([predicted_index])[0]
    gender = gender_from_speaker(speaker)

    return speaker, gender, confidence


print("Using:", "Mel Spectrogram + CNN" if BEST_PIPELINE == "mel" else "MFCC + Random Forest")


In [ ]:
import re
import joblib
import numpy as np
import librosa
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

def _gender_label_from_path(audio_path):
    """Read M/F only to create training targets from the dataset labels."""
    parts = [str(p) for p in Path(audio_path).parts]
    name = Path(audio_path).stem

    candidates = parts + [name]
    for value in reversed(candidates):
        m = re.search(r'(?i)(?:^|[-_])([mf])(?:$|[-_])', value)
        if m:
            return m.group(1).upper()

    # Also support labels ending in -M / -F.
    for value in reversed(candidates):
        if re.search(r'(?i)(?:^|[-_])[mf]$', value):
            return value[-1].upper()

    return None


def extract_gender_features(audio_path, sr=16000):
    """Extract acoustic features from the audio only."""
    y, _ = librosa.load(audio_path, sr=sr, mono=True)

    if len(y) == 0:
        raise ValueError(f"Empty audio file: {audio_path}")

    # Remove leading/trailing silence where possible.
    y, _ = librosa.effects.trim(y, top_db=30)

    features = []

    # MFCC statistics
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # Mel-spectrogram statistics
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    features.extend(np.mean(log_mel, axis=1))
    features.extend(np.std(log_mel, axis=1))

    # Spectral features
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zero_crossing = librosa.feature.zero_crossing_rate(y)

    for arr in (
        spectral_centroid,
        spectral_bandwidth,
        spectral_rolloff,
        zero_crossing,
    ):
        features.append(float(np.mean(arr)))
        features.append(float(np.std(arr)))

    # Fundamental-frequency statistics.
    # This is acoustic information from the voice, not a filename label.
    try:
        f0, voiced_flag, _ = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
        )
        voiced_f0 = f0[np.isfinite(f0)]
        if len(voiced_f0) > 0:
            features.extend([
                float(np.mean(voiced_f0)),
                float(np.std(voiced_f0)),
                float(np.median(voiced_f0)),
                float(np.percentile(voiced_f0, 25)),
                float(np.percentile(voiced_f0, 75)),
            ])
        else:
            features.extend([0.0] * 5)
    except Exception:
        features.extend([0.0] * 5)

    return np.asarray(features, dtype=np.float32)


# Find audio files using the same dataset directory already used by the notebook.
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
gender_audio_files = [
    p for p in Path(DATASET_DIR).rglob("*")
    if p.is_file() and p.suffix.lower() in audio_extensions
]

gender_X = []
gender_y = []

print(f"Found {len(gender_audio_files)} audio files for gender training.")

for idx, audio_path in enumerate(gender_audio_files, start=1):
    label = _gender_label_from_path(audio_path)

    # Only files for which the existing dataset naming convention
    # provides M/F are used to construct the training target.
    if label not in {"M", "F"}:
        continue

    try:
        gender_X.append(extract_gender_features(audio_path))
        gender_y.append(label)
    except Exception as e:
        print(f"Skipping {audio_path}: {e}")

gender_X = np.asarray(gender_X, dtype=np.float32)
gender_y = np.asarray(gender_y)

if len(gender_X) == 0:
    raise RuntimeError(
        "No M/F-labelled training files were found. "
        "The gender training cell needs dataset paths whose speaker labels "
        "contain -M or -F."
    )

print("Gender training samples:", len(gender_y))
print("Gender feature dimension:", gender_X.shape[1])
print("Gender label counts:", dict(zip(*np.unique(gender_y, return_counts=True))))

saved_gender_label_encoder = LabelEncoder()
gender_targets = saved_gender_label_encoder.fit_transform(gender_y)

saved_gender_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

saved_gender_model.fit(gender_X, gender_targets)

MODEL_DIR = Path(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    saved_gender_model,
    MODEL_DIR / "voice_gender_random_forest.joblib"
)

joblib.dump(
    saved_gender_label_encoder,
    MODEL_DIR / "gender_label_encoder.joblib"
)

print("Voice-based gender model trained and saved.")

In [ ]:
# Added Feature — Load saved models and final predictor
# ---------------------------------------------------------
# This cell does NOT retrain the speaker model or gender model.
# Run the gender-training cell above ONCE if the gender model
# has not already been created.

import joblib
from pathlib import Path

MODEL_DIR = Path(MODEL_DIR)

# Existing speaker model(s), saved by the original notebook.
# These names are loaded defensively from the existing notebook's
# model directory.
if "saved_mfcc_model" not in globals():
    mfcc_candidates = [
        MODEL_DIR / "mfcc_random_forest.joblib",
        MODEL_DIR / "mfcc_model.joblib",
    ]
    for p in mfcc_candidates:
        if p.exists():
            saved_mfcc_model = joblib.load(p)
            break

if "saved_label_encoder" not in globals():
    encoder_candidates = [
        MODEL_DIR / "speaker_label_encoder.joblib",
        MODEL_DIR / "label_encoder.joblib",
    ]
    for p in encoder_candidates:
        if p.exists():
            saved_label_encoder = joblib.load(p)
            break

# Gender model trained above.
if "saved_gender_model" not in globals():
    saved_gender_model = joblib.load(
        MODEL_DIR / "voice_gender_random_forest.joblib"
    )

if "saved_gender_label_encoder" not in globals():
    saved_gender_label_encoder = joblib.load(
        MODEL_DIR / "gender_label_encoder.joblib"
    )


def predict_speaker(audio_path):
    # -----------------------------
    # 1. Speaker identification
    # -----------------------------
    if BEST_PIPELINE == "mfcc":
        features = extract_mfcc_features(audio_path).reshape(1, -1)

        probabilities = saved_mfcc_model.predict_proba(features)[0]
        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])

        speaker = saved_label_encoder.inverse_transform(
            [predicted_index]
        )[0]

    else:
        mel = extract_mel_array(audio_path)
        probabilities = saved_mel_model.predict(
            np.expand_dims(mel, axis=0),
            verbose=0
        )[0]

        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])

        speaker = saved_label_encoder.inverse_transform(
            [predicted_index]
        )[0]

    # -----------------------------
    # 2. Gender identification
    # -----------------------------
    # IMPORTANT:
    # No speaker-name suffix is inspected here.
    # Gender is predicted from acoustic features extracted
    # directly from the audio.
    gender_features = extract_gender_features(audio_path).reshape(1, -1)

    gender_probabilities = saved_gender_model.predict_proba(
        gender_features
    )[0]

    gender_index = int(np.argmax(gender_probabilities))

    gender = saved_gender_label_encoder.inverse_transform(
        [gender_index]
    )[0]

    gender_confidence = float(
        gender_probabilities[gender_index]
    )

    return speaker, gender, confidence


print("Final predictor is ready.")
print("Gender is predicted from the speaker's voice.")


In [ ]:
# Cell 24 — Test the final predictor

speaker, gender, confidence = predict_speaker(audio_files[0])

print("Audio:", audio_files[0])
print("Predicted speaker:", speaker)
print("Gender:", gender)
print("Confidence:", f"{confidence:.2%}")


In [ ]:
# Cell 25 — Gradio UI

import gradio as gr

def gradio_predict(audio):
    if audio is None:
        return "No audio provided", "Unknown", "0%"

    speaker, gender, confidence = predict_speaker(audio)

    return speaker, gender, f"{confidence:.2%}"


with gr.Blocks(title="Amharic Voice Recognition") as demo:

    gr.Markdown(
        f"""
        # 🎤 Amharic Voice Recognition

        **Selected pipeline:** `{BEST_PIPELINE.upper()}`

        Upload a WAV/audio recording or record your voice.
        """
    )

    audio_input = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="Upload or Record Audio"
    )

    predict_button = gr.Button(
        "🔍 Recognize Speaker",
        variant="primary"
    )

    speaker_output = gr.Textbox(label="Speaker")
    gender_output = gr.Textbox(label="Gender")
    confidence_output = gr.Textbox(label="Confidence")

    predict_button.click(
        fn=gradio_predict,
        inputs=audio_input,
        outputs=[
            speaker_output,
            gender_output,
            confidence_output
        ]
    )

    gr.Markdown(
        """
        **Note:** This system identifies speakers that were included in
        the training dataset. It is not intended for security or identity verification.
        """
    )

demo.launch()
